# Bonsai 27B Q1: Colab T4 + prebuilt CUDA server + secure ngrok API (fixed)

Fixes applied versus the uploaded draft:

1. Model copy cell used a shell `!rsync` line with `$VAR` expansion mixed into Python — replaced with a checked `subprocess.run(...)` call that also deletes a stale/partial local copy before recopying.
2. Startup used f16 KV cache at ctx 8192, which overflows the T4's ~15 GiB VRAM once weights, KV, and compute buffers are counted — added `--flash_attn` + `--cache_type_k q8_0 --cache_type_v q8_0`, and an automatic context fallback (8192 -> 4096) if the server crashes during load.
3. Port cleanup only killed listeners; orphan `llama_cpp.server` processes without a live socket survived reruns — cleanup now also kills matching processes.
4. Readiness probe now retries on *any* connection error, checks `/health` and `/docs`, and dumps the server log on failure.
5. Every cell is now self-sufficient for its own imports, so out-of-order reruns do not raise `NameError`.

Prerequisites: GPU runtime selected in Colab, the GGUF in Google Drive at the configured path, and a free ngrok authtoken.


In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
import sys; print(sys.version)


In [ ]:
# Binary packages only. --only-binary prevents a hidden source compile fallback.
%pip -q uninstall -y llama-cpp-python
%pip -q install --only-binary=:all: --upgrade --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 'llama-cpp-python[server]' pyngrok httpx


In [ ]:
# Fail early if pip installed a CPU-only package.
from llama_cpp import llama_supports_gpu_offload
assert llama_supports_gpu_offload(), 'CUDA GPU offload unavailable. Restart with a GPU runtime, then rerun.'
print('Prebuilt CUDA llama-cpp-python wheel verified.')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# FIXED: pure-Python copy, no shell magic; stale/partial local files are removed first.
from pathlib import Path
import subprocess

DRIVE_MODEL_PATH = Path('/content/drive/MyDrive/models/Bonsai-27B-Q1_0.gguf')
LOCAL_MODEL_PATH = Path('/content/Bonsai-27B-Q1_0.gguf')

if not DRIVE_MODEL_PATH.is_file():
    raise FileNotFoundError(f'Model not found: {DRIVE_MODEL_PATH}')

expected_size = DRIVE_MODEL_PATH.stat().st_size
if LOCAL_MODEL_PATH.exists() and LOCAL_MODEL_PATH.stat().st_size != expected_size:
    print('Removing stale/partial local copy...')
    LOCAL_MODEL_PATH.unlink()

if not LOCAL_MODEL_PATH.is_file():
    subprocess.run(
        ['rsync', '-ah', '--info=progress2', str(DRIVE_MODEL_PATH), str(LOCAL_MODEL_PATH)],
        check=True,
    )
    assert LOCAL_MODEL_PATH.stat().st_size == expected_size, 'Copy size mismatch: copy again.'

print('Using:', LOCAL_MODEL_PATH)
print('Size GiB:', round(LOCAL_MODEL_PATH.stat().st_size / 1024**3, 2))


In [ ]:
from getpass import getpass

NGROK_AUTHTOKEN = getpass('Paste NGROK_AUTHTOKEN (hidden): ').strip()
API_KEY = getpass('Create/paste a strong API key, at least 20 characters (hidden): ').strip()

if not NGROK_AUTHTOKEN:
    raise ValueError('NGROK_AUTHTOKEN is required.')
if len(API_KEY) < 20:
    raise ValueError('API_KEY must contain at least 20 characters.')


In [ ]:
# CLEAN RESET: stop old listeners AND orphan llama_cpp.server processes.
import subprocess, time, socket

PORT = 8080
subprocess.run(['bash', '-lc', f'fuser -k {PORT}/tcp 2>/dev/null || true'], check=False)
subprocess.run(['bash', '-lc', "pkill -f 'llama_cpp.server' 2>/dev/null || true"], check=False)
time.sleep(4)

def port_is_free(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        return sock.connect_ex(('127.0.0.1', port)) != 0

if not port_is_free(PORT):
    listeners = subprocess.run(
        ['bash', '-lc', f'lsof -nP -iTCP:{PORT} -sTCP:LISTEN || true'],
        capture_output=True, text=True, check=False
    )
    raise RuntimeError(f'Port {PORT} is still occupied:\n{listeners.stdout}')

print(f'Port {PORT} is free.')


In [ ]:
# FIXED: q8_0 KV cache + flash attention to fit 27B on a 15 GiB T4, with context fallback.
from pathlib import Path
import subprocess, time, httpx

LOG_PATH = '/content/bonsai-server.log'

def build_command(ctx: int) -> list:
    return [
        'python', '-m', 'llama_cpp.server',
        '--model', str(LOCAL_MODEL_PATH),
        '--n_gpu_layers', '-1',
        '--n_ctx', str(ctx),
        '--n_batch', '512',
        '--flash_attn',
        '--cache_type_k', 'q8_0',
        '--cache_type_v', 'q8_0',
        '--host', '127.0.0.1',
        '--port', str(PORT),
        '--api_key', API_KEY,
    ]

def wait_ready(proc, timeout_s=240) -> bool:
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if proc.poll() is not None:
            return False
        for route in ('/health', '/docs'):
            try:
                if httpx.get(f'http://127.0.0.1:{PORT}{route}', timeout=3).status_code in (200, 401, 403):
                    return True
            except Exception:
                pass
        time.sleep(2)
    return False

server_process, CONTEXT_SIZE = None, None
for ctx in (8192, 4096):
    print(f'Trying n_ctx={ctx} ...')
    with open(LOG_PATH, 'w') as log_file:
        proc = subprocess.Popen(build_command(ctx), stdout=log_file, stderr=subprocess.STDOUT)
    if wait_ready(proc):
        server_process, CONTEXT_SIZE = proc, ctx
        print(f'Server ready on http://127.0.0.1:{PORT} (n_ctx={ctx})')
        break
    tail = Path(LOG_PATH).read_text(errors='replace')[-4000:]
    print(f'Startup failed at n_ctx={ctx}. Log tail:\n{tail}')

if server_process is None:
    raise RuntimeError('Model server failed to start at every context size. See log above.')


In [ ]:
# Confirm the loaded model uses T4 VRAM.
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
!tail -n 40 /content/bonsai-server.log


In [ ]:
# Local authenticated OpenAI-compatible test.
import json, httpx

HEADERS = {'Authorization': f'Bearer {API_KEY}'}

models_response = httpx.get(f'http://127.0.0.1:{PORT}/v1/models', headers=HEADERS, timeout=30)
models_response.raise_for_status()
models = models_response.json()
MODEL_ID = models['data'][0]['id']
print('MODEL_ID:', MODEL_ID)

payload = {
    'model': MODEL_ID,
    'messages': [{'role': 'user', 'content': 'Reply with exactly: Bonsai API is online.'}],
    'temperature': 0,
    'max_tokens': 32,
    'stream': False,
}

local_response = httpx.post(
    f'http://127.0.0.1:{PORT}/v1/chat/completions',
    headers=HEADERS, json=payload, timeout=300
)
local_response.raise_for_status()
local_result = local_response.json()
print(local_result['choices'][0]['message']['content'])
print(json.dumps(local_result.get('usage', {}), indent=2))


In [ ]:
# Publish only the API-key-protected server through a temporary HTTPS ngrok endpoint.
from pyngrok import ngrok

try:
    ngrok.kill()
except Exception:
    pass

ngrok.set_auth_token(NGROK_AUTHTOKEN)
tunnel = ngrok.connect(addr=PORT, proto='http')
PUBLIC_URL = tunnel.public_url.rstrip('/')
print('Public API base URL:', PUBLIC_URL + '/v1')


In [ ]:
# End-to-end external route verification.
import httpx

public_response = httpx.post(
    f'{PUBLIC_URL}/v1/chat/completions',
    headers=HEADERS, json=payload, timeout=300
)
public_response.raise_for_status()
print(public_response.json()['choices'][0]['message']['content'])
print('ngrok API verified.')


In [ ]:
print(f'''export OPENAI_BASE_URL="{PUBLIC_URL}/v1"
export OPENAI_API_KEY="YOUR_API_KEY"

curl -sS "$OPENAI_BASE_URL/chat/completions" \\
 -H "Authorization: Bearer $OPENAI_API_KEY" \\
 -H "Content-Type: application/json" \\
 -d '{{\
 "model": "{MODEL_ID}",\
 "messages": [{{"role": "user", "content": "Reply with exactly: Remote API works."}}],\
 "temperature": 0,\
 "max_tokens": 24\
 }}' | jq
''')


In [ ]:
# Shutdown when finished.
import subprocess
from pyngrok import ngrok

try:
    ngrok.disconnect(PUBLIC_URL)
    ngrok.kill()
except Exception:
    pass

if 'server_process' in globals() and server_process is not None and server_process.poll() is None:
    server_process.terminate()
    try:
        server_process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        server_process.kill()

print('Server and tunnel stopped.')
